# Verifying Tim's MutPred2 returns: missing + control variants

Checks the two files Tim sent back against the exact request that went out
(`AI_generated/mp2_request_for_tim.tsv` — 21,737 variants SJ was missing, plus 1,000
already-scored positive controls).

**Question 1 — coverage.** How many of the missing variants came back with a MutPred2 score?

**Question 2 — controls.** Do the 1,000 controls reproduce the scores already in
`MutPred2_for_filtered_variants_SJ.tsv`, the file holding MutPred2 scores for the variants
I extracted? Any drift there would mean the run itself is not comparable to the old scores.

Throughout, coverage means **scores we generated ourselves** — SJ's run plus Tim's. Tulika's
scores are never counted as coverage; her table appears only as provenance, to measure how much
of her set we now hold independently.

Self-contained: reads only files under `data/`, no dependency on other notebooks.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, VariantAnnotations_file

DATA             = Path(str(DATA_DIR))
AI_GENERATED_DIR = DATA / "AI_generated"

REQUEST    = AI_GENERATED_DIR / "mp2_request_for_tim.tsv"                      # what we asked for
TIM_RETURN = DATA / "MutPred2_scores" / "MutPred2_scores_missing_and_control.tsv"  # what came back
SJ_FILE    = DATA / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"    # the reference scores
ASD_RETURN = DATA / "MutPred2_scores" / "MutPred2_scores_ASD_brainspan.tsv"        # second file Tim sent

for f in (REQUEST, TIM_RETURN, SJ_FILE, ASD_RETURN, VariantAnnotations_file):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/mp2_request_for_tim.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_missing_and_control.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_for_filtered_variants_SJ.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_ASD_brainspan.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/filtered/variant_annotations.tsv


## 2. Load the request and Tim's return

The request is one row per variant. Tim's file is one row per **variant x transcript**, because
MutPred2 scores a substitution in the context of a specific protein isoform — so a single
genomic variant can come back with many scores.

A variant counts as *scored* if at least one of its transcript rows carries a non-null score.

In [2]:
request = pd.read_csv(REQUEST, sep="\t")
tim = (pd.read_csv(TIM_RETURN, sep="\t")
         .rename(columns={"ID": "Variant", "MutPred2 score": "MutPred2_tim"}))

MISSING  = set(request.loc[request["missing"], "Variant"])    # the 21,737 we needed scored
CONTROLS = set(request.loc[~request["missing"], "Variant"])   # the 1,000 positive controls

print(f"request      : {len(request):>6} variants  "
      f"({len(MISSING)} missing + {len(CONTROLS)} controls)")
print(f"Tim's return : {len(tim):>6} rows  "
      f"({tim['Variant'].nunique()} distinct variants, {tim['transcript'].nunique()} transcripts)")
print(f"               {tim['MutPred2_tim'].isna().sum()} rows carry a null score")

unexpected = set(tim["Variant"]) - set(request["Variant"])
print(f"\nvariants returned that were never requested : {len(unexpected)}")
assert not unexpected, "Tim's file contains variants outside the request"

request      :  22737 variants  (21737 missing + 1000 controls)
Tim's return :  51101 rows  (12829 distinct variants, 1892 transcripts)
               2 rows carry a null score

variants returned that were never requested : 0


## 3. Question 1 — coverage of the missing variants

In [3]:
scored_variants = set(tim.loc[tim["MutPred2_tim"].notna(), "Variant"])

missing_scored   = MISSING & scored_variants
missing_unscored = MISSING - scored_variants

print(f"missing variants requested : {len(MISSING):>6}")
print(f"  came back with a score   : {len(missing_scored):>6}  ({len(missing_scored)/len(MISSING):.1%})")
print(f"  no score                 : {len(missing_unscored):>6}  ({len(missing_unscored)/len(MISSING):.1%})")
print()
print(f"transcript-level rows returned for them : {len(tim[tim['Variant'].isin(MISSING)]):>6}")

missing variants requested :  21737
  came back with a score   :  11877  (54.6%)
  no score                 :   9860  (45.4%)

transcript-level rows returned for them :  45825


### 3a. Coverage by predicted consequence

The request deliberately included non-missense variants and left the filtering to Tim. MutPred2
only scores missense substitutions, so nonsense, synonymous and start-lost variants were never
expected to come back — they should not count against the coverage figure.

`no_name` is the ASD sheet, which carries no ClinVar `Name` column to parse; `unparsed` is a
`Name` whose protein consequence the regex could not read.

In [4]:
miss = request[request["missing"]].copy()
miss["scored"] = miss["Variant"].isin(scored_variants)

by_cons = miss.groupby("consequence")["scored"].agg(requested="size", scored="sum")
by_cons["unscored"]   = by_cons["requested"] - by_cons["scored"]
by_cons["pct_scored"] = (100 * by_cons["scored"] / by_cons["requested"]).round(1)

print(by_cons.sort_values("requested", ascending=False).to_string())

             requested  scored  unscored  pct_scored
consequence                                         
missense         12296   11869       427        96.5
nonsense          8499       0      8499         0.0
no_name            927       8       919         0.9
start_lost          15       0        15         0.0


### 3b. Missense coverage

The figure that actually matters: of the **missense** variants SJ was missing, how many are now
scored by our own run.

The `Tulika` column is **provenance only**. The goal of this request was to regenerate these
scores independently, so a variant Tulika happens to have is still a gap — her scores are not a
fallback and are not counted as coverage anywhere in this notebook. The breakdown below is
printed purely to show how much of her set we have now reproduced ourselves.

In [5]:
mis = miss[miss["consequence"] == "missense"]

print(f"missense variants requested : {len(mis):>6}")
print(f"  scored by Tim             : {mis['scored'].sum():>6}  ({mis['scored'].mean():.1%})")
print(f"  still a gap               : {(~mis['scored']).sum():>6}")
print()

still = miss[~miss["scored"]]
print("remaining gaps, by consequence (Tulika column is provenance only, not coverage):")
print(still.groupby(["consequence", "Tulika"]).size().rename("n").reset_index().to_string(index=False))

missense variants requested :  12296
  scored by Tim             :  11869  (96.5%)
  still a gap               :    427

remaining gaps, by consequence (Tulika column is provenance only, not coverage):
consequence  Tulika    n
   missense   False   52
   missense    True  375
    no_name   False  916
    no_name    True    3
   nonsense   False 8491
   nonsense    True    8
 start_lost   False   15


### 3c. Export the still-unscored list

In [6]:
out = AI_GENERATED_DIR / "variants_still_unscored_after_tim.tsv"
still.drop(columns=["scored"]).to_csv(out, sep="\t", index=False)

print(f"{len(still)} still-unscored variants written to\n  {out}")
still.drop(columns=["scored"]).head(10)

9860 still-unscored variants written to
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/variants_still_unscored_after_tim.tsv


,Variant,sheets,Tulika,SJ,missing,consequence
0,10_100806598_C/A,nonTarget,False,False,True,nonsense
1,10_100988226_C/T,nonTarget,False,False,True,nonsense
2,10_100989063_C/T,nonTarget,False,False,True,nonsense
3,10_101016407_G/A,nonTarget,False,False,True,nonsense
4,10_101021908_T/A,nonTarget,False,False,True,nonsense
5,10_102065829_G/T,nonTarget,False,False,True,nonsense
6,10_102066708_C/T,nonTarget,False,False,True,nonsense
7,10_102066987_C/T,nonTarget,False,False,True,nonsense
8,10_102067027_G/A,nonTarget,False,False,True,nonsense
9,10_102402284_C/T,nonTarget,False,False,True,nonsense


## 4. Question 2 — do the controls reproduce the SJ scores?

`MutPred2_for_filtered_variants_SJ.tsv` is keyed by `ID` (an ENST transcript) + `Substitution`,
with no genomic variant column. Tim returns `transcript` + `Substitution` + score. So the two
tables join directly on **(transcript, substitution)** — no coordinate mapping needed, and the
comparison is exactly like-for-like: same isoform, same amino-acid change.

In [7]:
sj = (pd.read_csv(SJ_FILE, sep="\t")
        .rename(columns={"ID": "transcript", "MutPred2": "MutPred2_sj"}))
sj = sj[sj["MutPred2_sj"].notna()]
assert not sj.duplicated(["transcript", "Substitution"]).any(), \
    "SJ file has duplicate transcript/substitution keys - the join would fan out"

tim_ctrl = tim[tim["Variant"].isin(CONTROLS)]

print(f"SJ reference entries          : {len(sj):>6}")
print()
print(f"control variants requested    : {len(CONTROLS):>6}")
print(f"  returned by Tim             : {tim_ctrl['Variant'].nunique():>6}")
print(f"  absent from Tim's return    : {len(CONTROLS - set(tim_ctrl['Variant'])):>6}")
print(f"control rows to compare       : {len(tim_ctrl):>6}")

SJ reference entries          : 154114

control variants requested    :   1000
  returned by Tim             :    952
  absent from Tim's return    :     48
control rows to compare       :   5276


### 4a. Score comparison

In [8]:
cmp = tim_ctrl.merge(sj, on=["transcript", "Substitution"], how="left")

matched = cmp[cmp["MutPred2_sj"].notna()].copy()
matched["abs_diff"] = (matched["MutPred2_tim"] - matched["MutPred2_sj"]).abs()

identical = int((matched["abs_diff"] == 0).sum())
differ    = int((matched["abs_diff"] > 0).sum())

print(f"control rows with an SJ counterpart : {len(matched):>6} of {len(cmp)}")
print(f"  score identical                   : {identical:>6}  ({identical/len(matched):.2%})")
print(f"  score differs at all              : {differ:>6}")
print(f"  largest absolute difference       : {matched['abs_diff'].max()}")
print()

disagree = matched[matched["abs_diff"] > 0]
if len(disagree):
    print(disagree[["Variant", "transcript", "Substitution", "MutPred2_tim", "MutPred2_sj", "abs_diff"]]
          .sort_values("abs_diff", ascending=False).head(20).to_string(index=False))
else:
    print("No control score disagrees with the SJ file.")

control rows with an SJ counterpart :   5260 of 5276
  score identical                   :   5260  (100.00%)
  score differs at all              :      0
  largest absolute difference       : 0.0

No control score disagrees with the SJ file.


### 4b. Control rows the SJ file has no entry for

Not disagreements — these are transcripts Tim scored that the SJ run never produced a score for,
so there is nothing to compare against. Listed in full because the count is small.

In [9]:
extra = cmp[cmp["MutPred2_sj"].isna()]

print(f"control rows with no SJ counterpart : {len(extra)} "
      f"({extra['Variant'].nunique()} variants, {extra['transcript'].nunique()} transcripts)")
print()
print(extra[["Variant", "transcript", "Substitution", "MutPred2_tim"]].to_string(index=False))

control rows with no SJ counterpart : 16 (16 variants, 3 transcripts)

        Variant        transcript Substitution  MutPred2_tim
2_105386507_G/T ENST00000447958.1          R4S           NaN
2_178566637_T/C ENST00000615779.5      T24858A         0.154
2_178569414_C/T ENST00000615779.5      R23932Q         0.213
2_178572013_C/T ENST00000615779.5      V23066M         0.315
2_178602342_C/T ENST00000615779.5      E16713K         0.516
2_178604074_T/C ENST00000615779.5      K16564E         0.572
2_178612104_G/T ENST00000615779.5      D15128E         0.187
2_178652533_C/G ENST00000615779.5      A11511P         0.105
2_178719230_T/A ENST00000615779.5       I7737L         0.176
2_178724471_C/A ENST00000615779.5       E6651D         0.312
2_178728921_C/T ENST00000615779.5       E6056K         0.406
2_178729030_A/T ENST00000615779.5       D6019E         0.244
2_178740323_A/T ENST00000615779.5       L3987M         0.123
2_178770209_C/T ENST00000615779.5       S2831N         0.234
2_178799649_T/

### 4c. The reverse direction — SJ entries Tim did not return

The comparison above only catches disagreements on pairs Tim returned. This checks the other
side: every SJ entry belonging to a control variant, and whether Tim returned it. SJ rows are
mapped to genomic variants through `variant_annotations.tsv`, the same mapping used to pick the
controls in the first place.

In [10]:
ann = (pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
         [["ENST", "aa_sub", "Variant"]].dropna().drop_duplicates())

sj_ctrl = ann.merge(sj, left_on=["ENST", "aa_sub"], right_on=["transcript", "Substitution"], how="inner")
sj_ctrl = sj_ctrl[sj_ctrl["Variant"].isin(CONTROLS)]

tim_keys = set(zip(tim_ctrl["transcript"], tim_ctrl["Substitution"]))
sj_ctrl["returned_by_tim"] = [k in tim_keys for k in zip(sj_ctrl["transcript"], sj_ctrl["Substitution"])]

print(f"SJ transcript-level entries for control variants : {len(sj_ctrl):>6}")
print(f"  also returned by Tim                           : {sj_ctrl['returned_by_tim'].sum():>6}")
print(f"  NOT returned by Tim                            : {(~sj_ctrl['returned_by_tim']).sum():>6}")
print(f"  control variants with no SJ entry returned     : "
      f"{len(CONTROLS - set(sj_ctrl.loc[sj_ctrl['returned_by_tim'], 'Variant'])):>6}")

SJ transcript-level entries for control variants :   3080
  also returned by Tim                           :   2726
  NOT returned by Tim                            :    354
  control variants with no SJ entry returned     :     48


### 4d. Controls Tim did not return at all

A control is by definition a variant SJ already scored, so anything absent here is a gap in the
new run rather than an unscorable variant.

In [11]:
absent = request[request["Variant"].isin(CONTROLS - set(tim["Variant"]))]

print(f"controls with no row in Tim's return : {len(absent)}")
print()
print(absent["consequence"].value_counts().to_string())
print()
print(absent[["Variant", "sheets", "Tulika", "consequence"]].head(20).to_string(index=False))

controls with no row in Tim's return : 48

consequence
missense      41
start_lost     4
no_name        2
nonsense       1

         Variant    sheets  Tulika consequence
  5_13902053_C/G       PCD    True    missense
  6_41282397_T/G nonTarget   False    missense
 3_180647081_G/A       PCD   False    missense
 10_27985211_G/C       PCD   False    missense
 19_55154206_T/A        CM    True    missense
  6_33441374_G/C nonTarget   False    missense
 14_23429777_T/G        CM    True    missense
  17_2672655_G/T nonTarget   False    missense
 6_112201608_C/T        CM    True    missense
10_110784432_G/T        CM    True    missense
  7_21558802_A/G       PCD    True    missense
  16_2102383_G/A       PKD    True    missense
 6_133274782_T/C        CM   False  start_lost
 7_117590353_A/C nonTarget   False    missense
  16_3738559_C/T nonTarget   False    missense
 17_31230381_A/G nonTarget   False    missense
  X_53554634_T/C nonTarget   False    missense
 X_101407903_T/C nonTarget   F

## 5. Where the independent score set stands now

The point of the request was to end up with a score set generated entirely by us — SJ's original
run plus Tim's new run — with **no scores taken from Tulika's files**. This section measures that
set against the full workbook (`extractedVariants.xlsx`, target + nonTarget sheets).

Tulika's table is read here only to answer "how much of what she has do we now hold
independently"; none of her scores enter the coverage counts.

In [12]:
from ExTSP.config import ExtractedVariants_file

SHEETS = ["CM", "PCD", "PH", "PKD", "ASD", "nonTarget"]
xl = pd.ExcelFile(ExtractedVariants_file)
sheets = {name: xl.parse(name) for name in SHEETS}

workbook = set()
for name in SHEETS:
    workbook |= set(sheets[name]["Variant"].dropna().unique())

# SJ's own coverage, mapped from (ENST, substitution) onto genomic variants
SJ_SCORED = set(
    ann.merge(sj, left_on=["ENST", "aa_sub"], right_on=["transcript", "Substitution"], how="inner")
       ["Variant"].unique()
)
TIM_SCORED  = scored_variants              # anything Tim returned with a non-null score
INDEPENDENT = SJ_SCORED | TIM_SCORED       # our own scores only - no Tulika

print(f"workbook variants (target + nonTarget) : {len(workbook):>6}")
print(f"  scored by SJ                         : {len(workbook & SJ_SCORED):>6}")
print(f"  newly scored by Tim                  : {len(workbook & TIM_SCORED - SJ_SCORED):>6}")
print(f"  INDEPENDENT total (SJ + Tim)         : {len(workbook & INDEPENDENT):>6}"
      f"  ({len(workbook & INDEPENDENT)/len(workbook):.1%})")
print(f"  still unscored by us                 : {len(workbook - INDEPENDENT):>6}")

workbook variants (target + nonTarget) :  60183
  scored by SJ                         :  38446
  newly scored by Tim                  :  11877
  INDEPENDENT total (SJ + Tim)         :  50323  (83.6%)
  still unscored by us                 :   9860


### 5a. Independent coverage per sheet

In [13]:
rows = []
for name in SHEETS:
    v = set(sheets[name]["Variant"].dropna().unique())
    rows.append({
        "sheet":      name,
        "variants":   len(v),
        "SJ":         len(v & SJ_SCORED),
        "SJ_plus_Tim": len(v & INDEPENDENT),
        "gap":        len(v - INDEPENDENT),
    })

cov = pd.DataFrame(rows)
cov["pct_covered"] = (100 * cov["SJ_plus_Tim"] / cov["variants"]).round(1)
cov.loc[len(cov)] = {
    "sheet": "ALL (union)", "variants": len(workbook),
    "SJ": len(workbook & SJ_SCORED), "SJ_plus_Tim": len(workbook & INDEPENDENT),
    "gap": len(workbook - INDEPENDENT),
    "pct_covered": round(100 * len(workbook & INDEPENDENT) / len(workbook), 1),
}
print(cov.to_string(index=False))

      sheet  variants    SJ  SJ_plus_Tim  gap  pct_covered
         CM     27540 13861        25084 2456         91.1
        PCD      8828  7498         7929  899         89.8
         PH       414   281          340   74         82.1
        PKD      2263  1571         1719  544         76.0
        ASD      1425   498          506  919         35.5
  nonTarget     19718 14739        14750 4968         74.8
ALL (union)     60183 38446        50323 9860         83.6


### 5b. How much of Tulika's set we now hold independently

The residual — variants she has a score for and we still do not — is the list that would have to
be rescored to drop her files entirely. Most of the workbook gap is nonsense and other
unscorable variants, which no run will ever fill; the number below is the part that matters.

In [14]:
TULIKA_RAW = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"

raw_t = pd.read_csv(TULIKA_RAW, sep="\t", low_memory=False)
TULIKA_SCORED = set(raw_t.loc[raw_t["mutpred"].notna(), "mutation"].dropna().unique())

overlap  = workbook & TULIKA_SCORED
residual = overlap - INDEPENDENT

print(f"workbook variants Tulika has a score for : {len(overlap):>6}")
print(f"  now reproduced independently           : {len(overlap & INDEPENDENT):>6}"
      f"  ({len(overlap & INDEPENDENT)/len(overlap):.1%})")
print(f"  still only in her files                : {len(residual):>6}")

out = AI_GENERATED_DIR / "variants_only_in_tulika.tsv"
request[request["Variant"].isin(residual)].to_csv(out, sep="\t", index=False)
print(f"\nresidual written to\n  {out}")

workbook variants Tulika has a score for :  36912
  now reproduced independently           :  36526  (99.0%)
  still only in her files                :    386

residual written to
  /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/variants_only_in_tulika.tsv


## 6. The second file Tim sent (`MutPred2_scores_ASD_brainspan.tsv`)

A separate delivery, not part of this request. Checked here only to see whether it closes any of
the gaps counted in section 3.

In [15]:
asd = pd.read_csv(ASD_RETURN, sep="\t").rename(columns={"MutPred2 score": "MutPred2_tim"})
asd["Variant"] = (asd["#CHROM"].astype(str) + "_" + asd["POS"].astype(str)
                  + "_" + asd["REF"] + "/" + asd["ALT"])

adds = set(asd.loc[asd["MutPred2_tim"].notna(), "Variant"]) & missing_unscored

print(f"ASD brainspan file : {len(asd)} rows, {asd['Variant'].nunique()} variants")
print(f"  rows overlapping this request                  : {asd['Variant'].isin(request['Variant']).sum()}")
print(f"  still-unscored requested variants it covers    : {len(adds)}")

ASD brainspan file : 1738 rows, 478 variants
  rows overlapping this request                  : 83
  still-unscored requested variants it covers    : 0


## 7. Summary

In [16]:
print("Q1  coverage of the missing variants")
print(f"    {len(missing_scored)} of {len(MISSING)} requested variants came back scored "
      f"({len(missing_scored)/len(MISSING):.1%})")
print(f"    restricted to missense, the only thing MutPred2 can score: "
      f"{mis['scored'].sum()} of {len(mis)} ({mis['scored'].mean():.1%})")
print(f"    still a gap: {len(missing_unscored)} variants "
      f"({(~mis['scored']).sum()} of them missense)")
print()
print("Q2  control reproducibility against MutPred2_for_filtered_variants_SJ.tsv")
print(f"    {len(matched)} transcript-substitution pairs compared")
print(f"    {identical} identical, {differ} different, max difference {matched['abs_diff'].max()}")
print(f"    {len(CONTROLS - set(tim['Variant']))} of {len(CONTROLS)} controls were not returned at all")
print()
print("Independent score set (SJ + Tim, no Tulika scores used)")
print(f"    covers {len(workbook & INDEPENDENT)} of {len(workbook)} workbook variants "
      f"({len(workbook & INDEPENDENT)/len(workbook):.1%})")
print(f"    reproduces {len(overlap & INDEPENDENT)} of the {len(overlap)} workbook variants "
      f"Tulika has ({len(overlap & INDEPENDENT)/len(overlap):.1%})")
print(f"    {len(residual)} workbook variants remain available only in her files")

assert (matched["abs_diff"] == 0).all(), \
    "a control score disagrees with MutPred2_for_filtered_variants_SJ.tsv"
print("\nPASS: every returned control score reproduces the SJ file exactly.")

Q1  coverage of the missing variants
    11877 of 21737 requested variants came back scored (54.6%)
    restricted to missense, the only thing MutPred2 can score: 11869 of 12296 (96.5%)
    still a gap: 9860 variants (427 of them missense)

Q2  control reproducibility against MutPred2_for_filtered_variants_SJ.tsv
    5260 transcript-substitution pairs compared
    5260 identical, 0 different, max difference 0.0
    48 of 1000 controls were not returned at all

Independent score set (SJ + Tim, no Tulika scores used)
    covers 50323 of 60183 workbook variants (83.6%)
    reproduces 36526 of the 36912 workbook variants Tulika has (99.0%)
    386 workbook variants remain available only in her files

PASS: every returned control score reproduces the SJ file exactly.
